# 🚀 VS#-15M Deep Neural Language Model Trainer (Kaggle T4 GPU)
### Continuous Training from Epoch 51 ➔ 100+ with Automatic Hugging Face Hub Sync

This notebook trains the **VS#-15M** (15 Million Parameters) deep neural language model on Kaggle's free GPU (Tesla T4 or P100).

**⚡ Features in this Comprehensive Version:**
- 🔄 **Auto-Resume from Hugging Face**: Automatically pulls your 50-epoch weights from [alexon-v8/VS-Sharp](https://huggingface.co/alexon-v8/VS-Sharp) and resumes training from **Epoch 51**!
- 🎯 **Top-P Nucleus Sampling**: Eliminates repetitive, garbled tokens during neural generation.
- ⚡ **FP16 Mixed Precision & AdamW**: Blazing fast CUDA execution (~3-4 seconds per epoch) with Cosine Annealing learning rate.
- ☁️ **Auto-Sync to Hugging Face**: Automatically pushes the new weights directly back to your Hugging Face repository upon training completion!

**⚠️ REQUIRED KAGGLE SETTINGS:**
- Right sidebar **Notebook Settings** ➔ **Accelerator**: `GPU T4 x2` (or `GPU P100`).
- Right sidebar **Notebook Settings** ➔ **Internet**: **`Internet on`** (required for GitHub & Hugging Face access).

## 1. Verify GPU Setup

In [ ]:
!nvidia-smi

## 2. Clone Repository & Pull 50-Epoch Weights from Hugging Face

In [ ]:
import os, urllib.request
%cd /kaggle/working
!rm -rf VS-Sharp
!git clone https://github.com/VerScript/VS-Sharp.git
%cd /kaggle/working/VS-Sharp

HF_URL = "https://huggingface.co/alexon-v8/VS-Sharp/resolve/main/model_weights.json"
print(f"[*] Pulling 50-epoch weights from Hugging Face: {HF_URL}...")
urllib.request.urlretrieve(HF_URL, "model_weights.json")
size_mb = os.path.getsize("model_weights.json") / (1024 * 1024)
print(f"✅ Successfully downloaded model_weights.json ({size_mb:.1f} MB)! Ready to resume from Epoch 51.")

## 3. Verify Master Dataset (1,142+ Polyglot & Syntax Pairs)

In [ ]:
import getpass, os
# Enter your Hugging Face write token (or set HF_TOKEN environment secret)
hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    hf_token = getpass.getpass('Enter your Hugging Face Write Token: ')

!pip install -q huggingface_hub
!python train_gpu.py \
    --epochs 50 \
    --batch-size 128 \
    --samples-per-epoch 15000 \
    --lr 0.0008 \
    --hf-token "$hf_token" \
    --hf-repo "alexon-v8/VS-Sharp"


## 4. Run Comprehensive 15M GPU Training (Epochs 51 ➔ 100+)
Runs FP16 mixed-precision training on CUDA, evaluates all 6 test categories after every epoch, and automatically uploads the updated checkpoint back to Hugging Face.

In [ ]:
!pip install -q huggingface_hub
!python train_gpu.py \
    --epochs 50 \
    --batch-size 128 \
    --samples-per-epoch 15000 \
    --lr 0.0008 \
    --hf-token "your_huggingface_token_here" \
    --hf-repo "alexon-v8/VS-Sharp"

## 5. Visualize Loss & Accuracy Curves

In [ ]:
import json
import matplotlib.pyplot as plt

with open('epoch_eval_history.json', 'r') as f:
    history = json.load(f)

epochs = [h['epoch'] for h in history]
losses = [h['loss'] for h in history]
accuracies = [h['accuracy'] * 100 for h in history]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, losses, 'r-o', label='Cross-Entropy Loss')
plt.title('VS#-15M Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, accuracies, 'b-o', label='Accuracy')
plt.title('VS#-15M Accuracy Curve')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

## 6. Interactive Prompt Tester with Top-P Nucleus Sampling

In [ ]:
import torch
from train_gpu import VSSharp15M, generate_text, calculate_hidden_size

with open('model_weights.json', 'r') as f:
    w_data = json.load(f)

vocab = w_data['vocab']
vocab_map = {t: idx for idx, t in enumerate(vocab)}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

hidden_size_1 = calculate_hidden_size(len(vocab))
model = VSSharp15M(len(vocab), hidden_size_1).to(device)

w = w_data['weights']
model.embedding.weight.data.copy_(torch.tensor(w['E'], dtype=torch.float32))
model.fc1.weight.data.copy_(torch.tensor(w['W1'], dtype=torch.float32).t())
model.fc1.bias.data.copy_(torch.tensor(w['b1'], dtype=torch.float32))
model.fc2.weight.data.copy_(torch.tensor(w['W2'], dtype=torch.float32).t())
model.fc2.bias.data.copy_(torch.tensor(w['b2'], dtype=torch.float32))
model.fc_out.weight.data.copy_(torch.tensor(w['W3'], dtype=torch.float32).t())
model.fc_out.bias.data.copy_(torch.tensor(w['b3'], dtype=torch.float32))

test_prompts = [
    "hello",
    "write a hello world program in verscript",
    "how to run python in verscript",
    "write javascript async fetch function",
    "write c code for binary search"
]

print("=== Testing VS#-15M Live Generation ===")
for p in test_prompts:
    res = generate_text(model, vocab, vocab_map, p, device, max_tokens=55, temperature=0.7, top_p=0.9)
    print(f"\n[Prompt]: {p}")
    print(f"[Generated Response]:\n{res}")

## 7. Direct 1-Click Browser Download (Kaggle Output)

In [ ]:
!cp /kaggle/working/VS-Sharp/model_weights.json.gz /kaggle/working/
!cp /kaggle/working/VS-Sharp/epoch_eval_history.json /kaggle/working/
print('✅ Copied weights to /kaggle/working/')
print('👉 Look at the right sidebar under Data > Output to download model_weights.json.gz directly!')